In [47]:
# Import pandas
import pandas as pd

In [ ]:
# Importing data for analysis
df = pd.read_csv("forex_data.csv")

In [49]:
# Copy of original data
df_original = df.copy()

In [50]:
df.head()

,slug,date,open,high,low,close,currency
0,GBP/EGP,2001-04-10,5.5809,5.5947,5.5947,5.5947,EGP
1,GBP/EGP,2001-06-04,5.4752,5.4939,5.4939,5.4939,EGP
2,GBP/EGP,2001-08-01,5.6799,5.6543,5.6543,5.6543,EGP
3,GBP/EGP,2002-07-29,7.2170,7.2170,7.2170,7.2170,EGP
4,GBP/EGP,2003-01-02,7.4243,7.3899,7.3899,7.3899,EGP


In [51]:
# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

In [52]:
# Sort values
df = df.sort_values(by=['slug', 'date'])

df.head()


,slug,date,open,high,low,close,currency
1387312,AUD/ARS,2007-05-29,2.5278,2.5296,2.5205,2.5259,ARS
1387313,AUD/ARS,2007-05-30,2.5261,2.5344,2.5159,2.5310,ARS
1387314,AUD/ARS,2007-05-31,2.5308,2.5523,2.5290,2.5471,ARS
1387315,AUD/ARS,2007-06-01,2.5483,2.5586,2.5440,2.5544,ARS
1387316,AUD/ARS,2007-06-04,2.5576,2.5619,2.5376,2.5600,ARS


In [53]:
# Check the duplicate rows
print(df.duplicated().sum()) 
df = df.drop_duplicates()

0


In [54]:
# Count number of records per currency pair
currency_counts = df.groupby('slug').size().reset_index(name='count')
print(currency_counts)

        slug  count
0    AUD/ARS   3676
1    AUD/BRL   3552
2    AUD/CAD   4602
3    AUD/CHF   4573
4    AUD/CNY   4584
..       ...    ...
335  USD/VND   4600
336  USD/XOF   4593
337  USD/XPF   4600
338  USD/ZAR   4596
339  USD/ZMW   2234

[340 rows x 2 columns]


In [55]:
# Calculate returns for all pairs
df['returns'] = df.groupby('slug')['close'].pct_change()

In [56]:
# Calculating volatility for each pair
df['volatility_30'] = df.groupby('slug')['returns'].rolling(30).std().reset_index(level=0, drop=True)

In [57]:
# Buy units at the FIRST closing price of each pair, then track value over time
investment = 100000

df['entry_price'] = df.groupby('slug')['close'].transform('first')
df['units'] = investment / df['entry_price']  
df['value'] = df['units'] * df['close']        
df['pnl'] = df.groupby('slug')['value'].transform(lambda x: x - x.iloc[0])

print(df[['slug', 'date', 'close', 'units', 'value', 'pnl']].head(10))

            slug       date  close      units       value       pnl
1387312  AUD/ARS 2007-05-29 2.5259 39589.8492 100000.0000    0.0000
1387313  AUD/ARS 2007-05-30 2.5310 39589.8492 100201.9082  201.9082
1387314  AUD/ARS 2007-05-31 2.5471 39589.8492 100839.3048  839.3048
1387315  AUD/ARS 2007-06-01 2.5544 39589.8492 101128.3107 1128.3107
1387316  AUD/ARS 2007-06-04 2.5600 39589.8492 101350.0139 1350.0139
1387317  AUD/ARS 2007-06-05 2.5687 39589.8492 101694.4455 1694.4455
1387318  AUD/ARS 2007-06-06 2.5828 39589.8492 102252.6624 2252.6624
1387319  AUD/ARS 2007-06-07 2.5926 39589.8492 102640.6429 2640.6429
1387320  AUD/ARS 2007-06-08 2.5931 39589.8492 102660.4379 2660.4379
1387321  AUD/ARS 2007-06-11 2.5886 39589.8492 102482.2835 2482.2835


In [58]:
# OHLC Features
df['range'] = df['high'] - df['low']
df['price_change'] = df['close'] - df['open']

In [59]:
# Cumulative return (how much has the investment grown over time)
df['cumulative_return'] = df.groupby('slug')['returns'].transform(
    lambda x: (1 + x).cumprod() - 1
)

In [60]:
# Show top performing pairs by final cumulative return
pd.set_option('display.float_format', '{:.4f}'.format)
top_returns = df.groupby('slug')['cumulative_return'].last().sort_values(ascending=False)
print('Top 10 Best Performing Pairs:')
print(top_returns.head(10))

Top 10 Best Performing Pairs:
slug
USD/IQD   4880.3151
USD/MMK    264.9842
EUR/MMK    245.4990
USD/SDG    219.0175
USD/ARS     96.7587
JPY/ARS     34.0292
EUR/ARS     28.5984
AUD/ARS     27.2183
GBP/ARS     24.1362
GBP/CUP     21.8217
Name: cumulative_return, dtype: float64


In [61]:
# Show lowest performing pairs by final cumulative return
print('Bottom 10 Worst Performing Pairs:')
print(top_returns.tail(10))

Bottom 10 Worst Performing Pairs:
slug
INR/NZD   -0.4405
INR/TWD   -0.4739
INR/THB   -0.4923
INR/CNY   -0.5134
INR/CHF   -0.5605
USD/MGA   -0.6257
USD/SOS   -0.7750
JPY/CHF   -0.9930
GBP/TRY   -0.9956
JPY/ILS   -0.9992
Name: cumulative_return, dtype: float64


In [ ]:
# Export raw untouched data
df_original.to_csv('forex_original.csv', index=False)
print("Original data exported!")

✅ Original data exported!


In [63]:
# Exporting the dataset
df.to_csv("forex_processed.csv", index=False)
print("Saved!")

Saved!


In [ ]:
# Export grouped summary — count, avg return, avg volatility per pair
summary = df.groupby('slug').agg(
    record_count   = ('close', 'count'),
    avg_close      = ('close', 'mean'),
    avg_volatility = ('volatility_30', 'mean'),
    cumulative_ret = ('cumulative_return', 'last'),
    final_pnl      = ('pnl', 'last')
).round(4).reset_index()

summary.to_csv('forex_summary.csv', index=False)
print("Summary report exported!")
print(summary.head())

✅ Summary report exported!
      slug  record_count  avg_close  avg_volatility  cumulative_ret  \
0  AUD/ARS          3676    14.8711          0.0105         27.2183   
1  AUD/BRL          3552     2.3294          0.0099          1.3742   
2  AUD/CAD          4602     0.9520          0.0056         -0.0307   
3  AUD/CHF          4573     0.8430          0.0072         -0.2915   
4  AUD/CNY          4584     5.6301          0.0093         -0.2148   

     final_pnl  
0 2721833.9206  
1  137418.3417  
2   -3072.0428  
3  -29154.4850  
4  -21475.0578  


In [ ]:
# Verify all exports worked
import os

files = ['forex_original.csv', 'forex_processed.csv', 'forex_summary.csv']
for f in files:
    size = os.path.getsize(f) / (1024*1024)  # size in MB
    print(f"{f} — {size:.2f} MB")

📁 forex_original.csv — 79.15 MB
📁 forex_processed.csv — 303.04 MB
📁 forex_summary.csv — 0.02 MB
